# Device Control AI Tutorial

This tutorial demonstrates how to use the Device Control AI component to manage and control quantum devices in a distributed environment.

In [ ]:
# Install required packages
!pip install paho-mqtt numpy matplotlib

In [ ]:
import json
import time
import paho.mqtt.client as mqtt
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Callable
from enum import Enum
import numpy as np
import matplotlib.pyplot as plt

## 1. Device Control Architecture

In [ ]:
class DeviceType(Enum):
    QUANTUM_PROCESSOR = "quantum_processor"
    CRYOGENIC_SYSTEM = "cryogenic_system"
    CONTROL_ELECTRONICS = "control_electronics"
    CLASSICAL_COMPUTER = "classical_computer"
    SENSOR = "sensor"

@dataclass
class DeviceState:
    device_id: str
    device_type: DeviceType
    status: str = "offline"
    parameters: Optional[Dict] = None
    last_updated: float = 0.0
    
    def update(self, new_state: Dict):
        """Update device state with new values."""
        for key, value in new_state.items():
            if hasattr(self, key):
                setattr(self, key, value)
        self.last_updated = time.time()
    
    def to_dict(self):
        """Convert device state to dictionary."""
        state = asdict(self)
        # Convert Enum to string
        state['device_type'] = self.device_type.value
        return state

## 2. MQTT Communication Layer

In [ ]:
import json
import paho.mqtt.client as mqtt
from typing import Callable, Dict


class MQTTClient:
    def __init__(self, client_id: str, broker: str = "localhost", port: int = 1883):
        self.client = mqtt.Client(client_id=client_id)
        self.broker = broker
        self.port = port
        self.connected = False
        self.subscriptions = {}

        # Set up callbacks
        self.client.on_connect = self._on_connect
        self.client.on_message = self._on_message
        self.client.on_disconnect = self._on_disconnect

    def connect(self):
        """Connect to the MQTT broker."""
        try:
            self.client.connect(self.broker, self.port, 60)
            self.client.loop_start()
            self.connected = True
            print(f"Connected to MQTT broker at {self.broker}:{self.port}")
        except Exception as e:
            print(f"Failed to connect to MQTT broker: {e}")

    def disconnect(self):
        """Disconnect from the MQTT broker."""
        if self.connected:
            self.client.loop_stop()
            self.client.disconnect()
            self.connected = False
            print("Disconnected from MQTT broker")

    def publish(self, topic: str, payload: Dict):
        """Publish a message to a topic."""
        if not self.connected:
            print("Not connected to MQTT broker")
            return False

        try:
            result = self.client.publish(
                topic,
                payload=json.dumps(payload),
                qos=1,
                retain=True
            )
            # Wait for publish to complete
            result.wait_for_publish()
            return result.rc == mqtt.MQTT_ERR_SUCCESS
        except Exception as e:
            print(f"Failed to publish message: {e}")
            return False

    def subscribe(self, topic: str, callback: Callable):
        """Subscribe to a topic with a callback function."""
        if not self.connected:
            print("Not connected to MQTT broker")
            return False

        try:
            self.client.subscribe(topic, qos=1)
            self.subscriptions[topic] = callback
            print(f"Subscribed to {topic}")
            return True
        except Exception as e:
            print(f"Failed to subscribe to {topic}: {e}")
            return False

    def _on_connect(self, client, userdata, flags, rc):
        if rc == 0:
            print("Connected to MQTT broker")
            self.connected = True
            
            # Resubscribe to topics
            for topic in self.subscriptions:
                client.subscribe(topic)
        else:
            print(f"Failed to connect to MQTT broker with code {rc}")

    def _on_message(self, client, userdata, msg):
        try:
            payload = json.loads(msg.payload.decode())
            
            # Call the appropriate callback
            callback = self.subscriptions.get(msg.topic)
            if callback:
                callback(msg.topic, payload)
            else:
                # Check for pattern-based subscriptions
                for topic, cb in self.subscriptions.items():
                    if mqtt.topic_matches_sub(topic, msg.topic):
                        cb(msg.topic, payload)
        except json.JSONDecodeError:
            print(f"Received invalid JSON on {msg.topic}")
        except Exception as e:
            print(f"Error processing message: {e}")

    def _on_disconnect(self, client, userdata, rc):
        print(f"Disconnected from MQTT broker (code: {rc})")
        self.connected = False

## 3. AI-Powered Device Controller

In [ ]:
import time
from typing import Dict, List, Optional
from .device_models import DeviceType, DeviceState  # Assuming these are defined elsewhere


class AIDeviceController:
    def __init__(self, client_id: str, broker: str = "localhost"):
        self.mqtt = MQTTClient(client_id, broker)
        self.devices: Dict[str, DeviceState] = {}
        self.rules: List[Dict] = []
    
    def start(self):
        """Start the device controller."""
        # Connect to MQTT broker
        self.mqtt.connect()
        
        # Subscribe to device status updates
        self.mqtt.subscribe("devices/+/status", self._handle_device_update)
        
    def stop(self):
        """Stop the device controller."""
        self.mqtt.disconnect()
    
    def register_device(self, device_id: str, device_type: DeviceType):
        """Register a new device with the controller."""
        if device_id not in self.devices:
            self.devices[device_id] = DeviceState(device_id, device_type)
            print(f"Registered device {device_id} of type {device_type.value}")
            return True
        else:
            print(f"Device {device_id} already registered")
            return False
    
    def add_rule(self, rule: Dict):
        """Add a rule for device control."""
        self.rules.append(rule)
    
    def send_command(self, device_id: str, command: str, parameters: Dict = None):
        """Send a command to a device."""
        if device_id not in self.devices:
            print(f"Device {device_id} not registered")
            return False
        
        message = {
            "command": command,
            "timestamp": time.time()
        }
        
        if parameters:
            message["parameters"] = parameters
        
        return self.mqtt.publish(f"devices/{device_id}/commands", message)
    
    def get_device_status(self, device_id: str) -> Optional[Dict]:
        """Get the status of a device."""
        if device_id in self.devices:
            return self.devices[device_id].to_dict()
        return None
    
    def _handle_device_update(self, topic: str, payload: Dict):
        """Handle device status update messages."""
        # Extract device_id from topic (format: devices/{device_id}/status)
        parts = topic.split('/')
        if len(parts) == 3 and parts[0] == "devices" and parts[2] == "status":
            device_id = parts[1]
            
            if device_id in self.devices:
                # Update existing device
                self.devices[device_id].update(payload)
                print(f"Updated device {device_id} status: {self.devices[device_id].status}")
                
                # Check rules
                self._evaluate_rules(device_id)
            else:
                # Auto-register new device if type is provided
                if "device_type" in payload:
                    try:
                        device_type = DeviceType(payload["device_type"])
                        self.register_device(device_id, device_type)
                        self.devices[device_id].update(payload)
                    except ValueError:
                        print(f"Invalid device type: {payload.get('device_type')}")
    
    def _evaluate_rules(self, device_id: str):
        """Evaluate rules for a device and take actions."""
        if device_id not in self.devices:
            return
        
        device = self.devices[device_id]
        
        for rule in self.rules:
            if self._rule_matches(rule, device):
                self._execute_action(rule["action"], device_id)
    
    def _rule_matches(self, rule: Dict, device: DeviceState) -> bool:
        """Check if a rule matches the current device state (STUB)"""
        # Implement actual rule matching logic here
        # Example: return device.status == rule["condition"]
        return False
    
    def _execute_action(self, action: Dict, device_id: str):
        """Execute an action for a device (STUB)"""
        # Implement actual action execution logic here
        # Example: self.send_command(device_id, action["command"], action["params"])
        print(f"Executing action for {device_id}: {action}")

In [ ]:
import time
from enum import Enum
from typing import Dict, Optional

# Assuming DeviceType is defined elsewhere, adding it for completeness
class DeviceType(Enum):
    QUANTUM_PROCESSOR = "quantum_processor"
    CRYOGENIC_SYSTEM = "cryogenic_system"

# Minimal DeviceState implementation for the example
class DeviceState:
    def __init__(self, device_id: str, device_type: DeviceType):
        self.device_id = device_id
        self.device_type = device_type
        self.status = "offline"
        self.parameters: Dict[str, float] = {}
    
    def update(self, payload: Dict):
        self.status = payload.get("status", self.status)
        if "parameters" in payload:
            self.parameters.update(payload["parameters"])
    
    def to_dict(self) -> Dict:
        return {
            "device_id": self.device_id,
            "device_type": self.device_type.value,
            "status": self.status,
            "parameters": self.parameters
        }

def main():
    # Create AI controller
    controller = AIDeviceController("ai-controller")
    controller.start()
    
    # Register devices
    controller.register_device("qpu-001", DeviceType.QUANTUM_PROCESSOR)
    controller.register_device("cryo-001", DeviceType.CRYOGENIC_SYSTEM)
    
    # Add a rule
    controller.add_rule({
        "condition": {
            "device_type": DeviceType.CRYOGENIC_SYSTEM.value,
            "parameters": {
                "temperature": {"above": 4.0}  # Temperature above 4K
            }
        },
        "action": {
            "type": "command",
            "target": "qpu-001",
            "command": "shutdown",
            "reason": "Temperature too high"
        }
    })
    
    # Simulate a temperature increase
    controller.mqtt.publish("devices/cryo-001/status", {
        "device_type": "cryogenic_system",
        "status": "online",
        "parameters": {
            "temperature": 4.5,  # Above threshold
            "pressure": 1.0
        }
    })
    
    # Wait for rules to be processed
    time.sleep(1)
    
    # Check device status
    print("QPU Status:", controller.get_device_status("qpu-001"))
    print("Cryo Status:", controller.get_device_status("cryo-001"))
    
    # Stop controller
    controller.stop()

if __name__ == "__main__":
    main()

## 4. AIDeviceController

In [ ]:
def _rule_matches(self, rule: Dict, device: DeviceState) -> bool:
    """Check if a rule's condition matches the device state."""
    condition = rule.get("condition", {})
    
    # Check device type
    if "device_type" in condition and condition["device_type"] != device.device_type.value:
        return False
    
    # Check parameters
    if "parameters" in condition and device.parameters:
        param_conditions = condition["parameters"]
        for param_name, conditions in param_conditions.items():
            if param_name not in device.parameters:
                return False
            
            param_value = device.parameters[param_name]
            
            # Check conditions
            if "above" in conditions and not param_value > conditions["above"]:
                return False
            if "below" in conditions and not param_value < conditions["below"]:
                return False
            if "equals" in conditions and not param_value == conditions["equals"]:
                return False
    
    return True

def _execute_action(self, action: Dict, device_id: str):
    """Execute an action based on a triggered rule."""
    action_type = action.get("type")
    
    if action_type == "command":
        target_id = action.get("target", device_id)
        command = action.get("command")
        reason = action.get("reason", "Rule triggered")
        
        if command:
            print(f"Executing command '{command}' on device {target_id} (Reason: {reason})")
            self.send_command(target_id, command, {"reason": reason})
    elif action_type == "alert":
        message = action.get("message", "Alert triggered")
        print(f"ALERT: {message}")
        # In a real system, you might send this to a notification service

In [ ]:
{
"metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}